In [ ]:
import sys
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import random
import numpy as np

sys.path.append('../')
from data_provider.data_loader import load_dataset

### Experiment Configuration

In [ ]:
# Data parameters
DATA_FOLDER = '../data/'
DATASET_NAME = 'train_v2_drcat_02' # Change to dataset you wish to train on

# Processing parameters
TEST_SIZE = 0.2
VAL_SIZE = 0.1
RANDOM_SEED = 2026 # Random seed for reproducibility
MAX_FEATURES = 5000 # Limit the number of features for TF-IDF Vectorization

# Model Hyperparameters
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 0.001

# Reproducibility
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


#### Load Dataset

In [98]:
df = load_dataset(DATA_FOLDER, DATASET_NAME)

# Check the column names to ensure features and labels are loaded correctly
print(df.head())

                                                text  label
0  Phones\n\nModern humans today are always on th...      0
1  This essay will explain if drivers should or s...      0
2  Driving while the use of cellular devices\n\nT...      0
3  Phones & Driving\n\nDrivers should not be able...      0
4  Cell Phone Operation While Driving\n\nThe abil...      0


#### Split dataset into training, validation, and test sets

In [99]:
# Dataset is not randomly ordered, so we will shuffle it to ensure a representative split between training and testing sets

train_val_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=df['label'],
    shuffle=True
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=(VAL_SIZE / (1.0 - TEST_SIZE)),
    random_state=RANDOM_SEED,
    stratify=train_val_df['label'],
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f'Train set size: {len(train_df)}')
print(f'Validation set size: {len(val_df)}')
print(f'Test set size: {len(test_df)}')

Train size: 31407
Validation size: 4487
Test size: 8974


#### Vectorize text data using TF-IDF

In [100]:
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)

X_train = vectorizer.fit_transform(train_df['text']).toarray()
X_val = vectorizer.transform(val_df['text']).toarray()
X_test = vectorizer.transform(test_df['text']).toarray()

y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

#### Logistic Regression Model

In [101]:
class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super(LogisticRegressionModel, self).__init__()
        self.linear = nn.Linear(input_dim, 1)
    
    def forward(self, x):
        return self.linear(x)

#### Training Loop for Logistic Regression Model

In [102]:
def run_logistic_regression(X_train, y_train, X_val, y_val, X_test, y_test):
    model = LogisticRegressionModel(X_train.shape[1])
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)

    num_epochs = EPOCHS
    batch_size = BATCH_SIZE

    for epoch in range(num_epochs):
        model.train()
        for i in tqdm(range(0, len(X_train_tensor), batch_size)):
            X_batch = X_train_tensor[i:i+batch_size]
            y_batch = y_train_tensor[i:i+batch_size]

            logits = model.forward(X_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            model.eval()
            with torch.no_grad():
                val_logits = model.forward(X_val_tensor)
                val_loss = criterion(val_logits, y_val_tensor)

                val_probabilities = torch.sigmoid(val_logits)
                val_predicted_labels = (val_probabilities >= 0.5).float()
                val_correct = (val_predicted_labels == y_val_tensor).sum()
                val_accuracy = val_correct / len(y_val_tensor)

        print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {loss.item()}, Validation Loss: {val_loss.item()}, Validation Accuracy: {val_accuracy.item()}')

    model.eval()
    with torch.no_grad():
        X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
        y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)
        logits = model.forward(X_test_tensor)
        probabilities = torch.sigmoid(logits)
        predicted_labels = (probabilities >= 0.5).float()

        correct = (predicted_labels == y_test_tensor).sum()
        accuracy = correct / len(y_test_tensor)
        print(f'Test Accuracy: {accuracy.item()}')

        y_true = y_test_tensor.flatten().numpy()
        y_pred = predicted_labels.flatten().numpy()
        print(classification_report(y_true, y_pred, target_names=['Human', 'AI']))

    return model


def predict_custom_texts(model, vectorizer, texts):
    if isinstance(texts, str):
        texts = [texts]

    features = vectorizer.transform(texts).toarray()
    feature_tensor = torch.tensor(features, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(feature_tensor)).squeeze(1).numpy()

    predictions = (probabilities >= 0.5).astype(int)
    results = pd.DataFrame(
        {
            'text': texts,
            'predicted_label': ['AI' if pred == 1 else 'Human' for pred in predictions],
            'ai_probability': probabilities,
        }
    )
    print(results)
    return results



#### Train our Logistic Regression model

In [103]:
trained_lr_model = run_logistic_regression(X_train, y_train, X_val, y_val, X_test, y_test)

100%|██████████| 982/982 [00:00<00:00, 1122.91it/s]


Epoch 1/10, Train Loss: 0.3180471360683441, Validation Loss: 0.30972015857696533, Validation Accuracy: 0.9396032691001892


100%|██████████| 982/982 [00:00<00:00, 1207.85it/s]


Epoch 2/10, Train Loss: 0.19394823908805847, Validation Loss: 0.1933044195175171, Validation Accuracy: 0.9692444801330566


100%|██████████| 982/982 [00:00<00:00, 1191.34it/s]


...

Epoch 8/10, Train Loss: 0.03777527064085007, Validation Loss: 0.05106993392109871, Validation Accuracy: 0.9908624887466431


100%|██████████| 982/982 [00:00<00:00, 1236.58it/s]


Epoch 9/10, Train Loss: 0.03252997249364853, Validation Loss: 0.04526657983660698, Validation Accuracy: 0.9917539358139038


100%|██████████| 982/982 [00:00<00:00, 1331.67it/s]

Epoch 10/10, Train Loss: 0.028561247512698174, Validation Loss: 0.040744598954916, Validation Accuracy: 0.9921996593475342
Test Accuracy: 0.9917539358139038
              precision    recall  f1-score   support

       Human       0.99      1.00      0.99      5474
          AI       1.00      0.98      0.99      3500

    accuracy                           0.99      8974
   macro avg       0.99      0.99      0.99      8974
weighted avg       0.99      0.99      0.99      8974



#### Test Logistic Regression with Custom Inputs

In [104]:
custom_texts = [
    "This essay argues that social media can improve civic participation when paired with strong moderation policies.",
    "In conclusion, the provided evidence clearly demonstrates a multifaceted and highly structured perspective on the topic.",
    "This isn't working very well, as it seems to just be predicting AI for longer messages and Human for shorter messages. At least it's with a lower probability.",
]

predict_custom_texts(trained_lr_model, vectorizer, custom_texts)


                                                text predicted_label  \
0  This essay argues that social media can improv...              AI   
1  In conclusion, the provided evidence clearly d...              AI   
2  This isn't working very well, as it seems to j...           Human   

   ai_probability  
0        0.997183  
1        0.955919  
2        0.456916  


,text,predicted_label,ai_probability
0,This essay argues that social media can improv...,AI,0.997183
1,"In conclusion, the provided evidence clearly d...",AI,0.955919
2,"This isn't working very well, as it seems to j...",Human,0.456916
